# Phase 3 — End-to-End Pipeline Execution & Pilot Evaluation
**Hallucination Detection & Repair in LLM-Generated Answers**

This notebook runs the full 5-stage pipeline on Google Colab using:
- **Groq API** (`llama-3.3-70b-versatile`)
- **Persistent SQLite Cache** on Google Drive (zero cost for duplicate prompts & safe against disconnects)
- **BM25 / FAISS Retrieval** from the Drive-stored index
- **Batch evaluation & delta scoring**

In [ ]:
# 1. Mount Drive, configure paths & API keys
from google.colab import drive, userdata
import os
from pathlib import Path

drive.mount('/content/drive')
DRIVE_BASE = Path("/content/drive/MyDrive/hallucination_project")

# Set Groq API key: from Colab Secrets or prompt
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except Exception:
    os.environ["GROQ_API_KEY"] = input("Enter your GROQ_API_KEY: ").strip()

# Point SQLite cache to Drive so calls are permanently cached
os.environ["LLM_CACHE_PATH"] = str(DRIVE_BASE / "llm_cache.db")
print(f"Cache set to: {os.environ['LLM_CACHE_PATH']}")

In [ ]:
# 2. Clone repo and install requirements
!git clone https://github.com/YOUR_USERNAME/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git repo
%cd repo
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm

In [ ]:
# 3. Run on a single interactive example
from src.schema import AnswerRecord
from src.pipeline import run_pipeline

index_path = str(DRIVE_BASE / "indexes" / "bm25_A.pkl")

record = AnswerRecord(
    id="colab_demo_001",
    source_dataset="manual",
    split="dev",
    question="When did Neil Armstrong walk on the Moon and who was with him?",
    answer="Neil Armstrong walked on the Moon on July 20, 1969. He was accompanied by John Glenn, who served as lunar module pilot."
)

print("Running pipeline...")
result = run_pipeline(record, index_path=index_path, provider="groq", verbose=True)

print("\n" + "="*60)
print(f"Original Answer:\n  {result.answer}")
print(f"\nExtracted Claims & Verdicts:")
for c in result.claims:
    print(f"  [{c.label or 'Unv':13s}] {c.text}")
    if c.label_rationale:
        print(f"    ↳ Reason: {c.label_rationale}")
print(f"\nRepaired Answer:\n  {result.repaired_answer}")
print(f"\nReliability Score: {result.reliability_score_before:.2f} ➔ {result.reliability_score_after:.2f}")
print("="*60)

In [ ]:
# 4. Pilot Batch Run (20 examples from HaluEval or FELM)
from src.pipeline import run_pipeline_batch

input_jsonl = DRIVE_BASE / "data" / "processed" / "halueval.jsonl"
output_jsonl = DRIVE_BASE / "results" / "phase3_pilot.jsonl"

records = []
if input_jsonl.exists():
    with open(input_jsonl) as f:
        for line in f:
            if line.strip():
                records.append(AnswerRecord.from_jsonl_line(line))
    pilot_records = [r for r in records if r.split == "dev"][:20]
    print(f"Loaded {len(pilot_records)} dev records for pilot run.")
    
    results = run_pipeline_batch(
        pilot_records,
        output_path=str(output_jsonl),
        index_path=index_path,
        provider="groq",
        log_every=5
    )
else:
    print(f"File {input_jsonl} not found. Run 01_download_data.ipynb first.")

In [ ]:
# 5. Evaluate Pilot Run Results
!python scripts/run_eval.py --results {output_jsonl}